# Team Fight Tactics Team Composition Analysis
#### By: Aaron Man Chun Li
#### For: Dr. Ben Winjum; 402625: Machine Learning (Python)COM SCI-X 450.4
#### DATE: JUNE, 9th, 2025

## Jupyter notebook:
Your notebook must include an introductory narrative that provides:

- a short description of your dataset
- why you are interested in it
- a link to where others can access the data
- a link to documentation about the data (if different from the data access link)
- an introductory summary of at least one published work that applied machine learning to the dataset

The notebook must then present a clear narrative showing a substantive and specific machine learning workflow that includes:

- Importing and organizing the data
- Basic statistical analysis and/or visualizations of the data
- to present a general overview of the data
- to process data as needed, such as dealing with missing values and applying transformations
- Clear description of which variables will be used for machine learning
- An application of at least thee machine learning algorithms to achieve the prediction objective
   - You must use XGBoost for one of the algorithms
   - You must use TensorFlow for one of the algorithms
   - Cross validation to learn the optimum values of hyperparameters
- Measurements of how well the learned models generalize to new or test data
- A comparison of your models' results against each other

The notebook must then end with:

- a discussion of the advantages and disadvantages of each algorithm
- an explanation of why the results may be different between them
- a comparison of your model(s) with prior work
- I will not grade you on whether your results are comparable to or better than other work! Simply give a thoughtful and well-reasoned explanation of how they make sense together (or don't).
- a discussion of how one must consider biases, interpretability, and ethics with respect to your dataset and any algorithms that have been applied to it
- conclusions about what you found to work best and what key points should be drawn from your work

The notebook should:

- Tell the story of your analysis in approximately 1500+ words (not including tables, figures, captions, or references).
- If applicable, it is perfectly ok for this notebook to describe how your analysis changed as you went through the machine learning process.

The top of your notebook must include your name(s), the date, and your project title.

# Introduction

![asdf](https://raiseyourgame.com/wp-content/uploads/2022/08/Andy_Teamfight_Podcast_Optional_Featured_sm_792x447.jpg)

https://teamfighttactics.leagueoflegends.com/en-us/

Team Fight Tactics (TFT) is an auto-battler stratgery game developed by Riot Games, local to west Los Angeles, set in the League of Legends universe. Players compete in 8-player free-for-all matches, where each round consists of automatic battles between squads of units that the player builds over time (hence the name auto-battler). The goail is to be the last player standing by strategically managing your economogy, positioning, and __most critcally__ your team composition. 

# Core Mechanics
- __Unit Drafting__ : Player acquire units from a shared pool using gold. Units have different origins and classes (aka traits).
- __Trait Synergies__ :Units with matching traits grant bonuses, making synergies the foundation of strong team compositions.
- __Economy Management__: Player earn gold each round, and must balance spending on units, leveling up, and saving for interest.
- __Positioning__: Units are placed on a grid, and while they act automatically in combat, smart placement can greatly affect outcomes—changing who gets targeted, how long key units survive, and whether abilities land effectively.
- __Augments & Items__: Augments add game-changing bonuses; where items crafted from components drastically empower unit performance.

# Data Set

Riot Games offers an API that allows developers to gather informatino about the game.
These information are avaiable here: https://developer.riotgames.com/
A Riot Games account will be required, and you'll be given immedate access to a developement API key that you can insert to run this code if you do choose to do so.

The key needs to go into a .env file, and the content of the file will look like: 
TFT_API_KEY="RGAPI-12341234-12341234-12341234-12341234-123412341234"

For TFT, there are a limited amount of information avaiable on the API.
The two key piece of information of interest are player match history, and their ranking.

Our dataset involves will involve a large number of matches ( greater 20k ) gathered through the API.

Each match have data about the _FINAL STATE_ of the game. 
- The units each playerd field, and the respective unit's items
- Synergy level hit per activate trait
- Number of players eliminated
- Damage dealt by each player
- Final Placements, Top 4 is considered a win
- Level
- Gold Remaining
- Rounds played

The match DOES NOT have data about:
- Augments
- Timeline and snapshots of the game
- Game modifiers

The RAW gathered data can be access via github here:
https://github.com/aaronmcli/TFTInsights/data

and imported and processed data will be submitted with this assigment on canvas, and the results can also be viewed at https://www.aaronmcli.com/TFTInsights

# Interest & Goal

As a gamer, my main interest is winning a game and climbing through the ranks. 
As a data science student, I'm interested in how to build compositions correlates to winning, and how to generate compositions that are powerful, and find potential dark horses. 

Our goal in our regression analysis will be:
Composition (Unit, Items, Traits) --> Placements 

Ultimately, I would like the apply the knowledge we would have gathered and, build a tool that can inform the player during their game how to best use their resouce. 
A way to learn the game via the data we have. 

# Data Analysis Process

## 1 - Gathering the data
Please see the notebook TFT_GameIngest.ipynb

We started the project by learning how to importn JSON files from the Riot API and slowly optmizing the code to be more time efficient - by pulling data closer to Riot's API call Limit, and by cacheing data whenever it make sense.

## 2 - Importing and organizing the data 
Please see the notebook TFT_DataImport.ipynb

When we started importing the data, we started research on how best to input the data for machine learning algorithm, and I concluded that one-hot like approach would best suit the needs of XGBoost and Tensorflow. 
After additional analysis, which we will discuss later in the notebook, we also noticed a potential for feature engineering to reduce the dataset and improve generalization of predictor of player success. 



# 3 - Exploratory Data Analysis
We conducted an initial exploration of the dataset to identify useful gameplay patterns. This included examining which units most frequently hold key items and identifying the most impactful traits contributing to success. Additionally, we reviewed player skill distribution to confirm that the dataset provides a representative sample of the broader player population.

# 4 - Placement Regression via Decision Tree


## Model Evaluation Metrics (Sorted by R²)

| Model           | MAE    | MSE    | R²     |
|----------------|--------|--------|--------|
| Trait + Unit   | 1.5534 | 3.5230 | 0.3269 |
| Units Only     | 1.5571 | 3.5369 | 0.3242 |
| Unit + Items   | 1.5639 | 3.5593 | 0.3199 |
| Trait Only     | 1.7821 | 4.4365 | 0.1523 |
| Items Only     | 1.8370 | 4.6879 | 0.1043 |

Based on our dataset, units are the strongest predictors of placement outcomes. Items, by contrast, add little predictive value. This is likely because players who reach top placements are already equipping optimal items on optimal units—especially at higher skill tiers. This relationship, while obvious to skilled players, is difficult for the ML model to infer independently.

When inspecting the decision tree trained on Traits + Units, we observed that the top decision nodes consistently featured 4 and 5-cost units, highlighting their outsized impact on placement. This suggests a strong opportunity for feature engineering, such as adding features like the count of units by cost tier (e.g., number of 1-cost, 2-cost, etc.) to better capture player board strength.

Overall, player bias is reflected in this behavior, as players naturally pursue optimal strategies to win, placing strong items on strong units. The model is capturing this, but additional feature work could help refine these relationships further.


# 4b - Placement Regression with XGBoost

### XGBoost Model Comparison

| Model Variant         | MAE    | MSE    | R² Score |
|-----------------------|--------|--------|----------|
| XGBoost (Base)        | 1.2044 | 2.2395 | 0.5727   |
| XGBoost with CV       | 1.1637 | 2.1431 | 0.5911   |
| XGBoost with LASSO    | 1.1493 | 2.0921 | 0.6008   |

XGBoost provides a more accurate representation of placement regression compared to the decision tree. It consistently confirms the significance of 4 and 5-cost units in determining player success.

Feature importance analysis from XGBoost highlights that players who manage to field multiple upgraded 4 and 5-cost units dramatically increase their chances of achieving higher placements. This aligns with gameplay expectations: these units are often win conditions in the meta, and players who stabilize into these units tend to perform better.

While XGBoost improves predictive accuracy, it still cannot account for unmeasured confounding factors like augments, board positioning, opponent strength, or player decision-making under uncertainty, which likely explain the remaining variance in placement predictions.



# 4c - Placaement Regression with Tensorflow

| Model Variant         | MAE    | MSE    | R² Score |
|-----------------------|--------|--------|----------|
| XGBoost (Base)        | 1.2044 | 2.2395 | 0.5727   |
| XGBoost with CV       | 1.1637 | 2.1431 | 0.5911   |
| XGBoost with LASSO    | 1.1493 | 2.0921 | 0.6008   |
| Deep Neural Network   | 1.0191 | 1.6745 | 0.6388   |

The Neural Network provided slightly better predictive performance than XGBoost, achieving the highest R² in this analysis. However, the improvement in predictive accuracy was marginal.

Unlike XGBoost, the neural network does not provide built-in feature importance, making it less interpretable for extracting actionable insights. While effective at capturing non-linear relationships, it offered limited additional understanding of what specific features drive player success.

Given the modest performance gain and lack of interpretability, the neural network was more of a predictive tool rather than a method for generating player-facing recommendations or design insights.

# 5 - Feature Engineering

With the results from Decision Tree and XGBoost consistently emphasizing the importance of 4- and 5-cost units, we applied feature engineering to better capture general board strength, combining unit and trait features. However, even with these engineered features, both models continued to prioritize individual high-cost units over trait compositions in their feature importances.

This suggests that, in this dataset, high-cost units serve as strong stand-alone predictors of player success, while traits appear secondary or redundant—likely because traits naturally synergize with these units when players are optimizing their boards. The analysis ultimately highlights the dominant role of established, high-cost units in determining player outcomes.

5 - Unit Recommendation 